In [1]:
from pathlib import Path
import os
import pandas as pd
from src.agent_profiles.skill_generator import get_project_root
import dspy
from tqdm import tqdm

# dataset_path = Path("~/mnt/shared-resources-sentient-research/salah_resources/datasets/officeqa/")
# csv_path = os.path.join(dataset_path, "officeqa.csv")

# train_data = pd.read_csv(csv_path)
# hard_data = train_data[train_data['difficulty'] == 'hard']
# easy_data = train_data[train_data['difficulty'] == 'easy']

In [2]:
#   - Lookup/Extraction: pull a single value/entity/page/count from a table/chart/PDF with minimal computation
#   - Aggregation: compute totals/means/geomeans/weighted averages over a set of reported values
#   - Comparison/Change: differences, percent changes/differences, ratios/shares, argmax/argmin month/year
#   - Transform/Normalize: inflation-adjust, currency-convert, log/Box-Cox, winsor/trim, build returns before summarizing
#   - Statistical Metrics: SD/variance/CV/MAD/z-score, correlation, inequality/tail/entropy indices (Gini/Theil/HHI/Zipf/Pareto/KL/skew/kurt)
#   - Modeling/Forecasting: OLS/polynomial regression, ARIMA/smoothing/HP filter, projections & forecast errors

In [3]:
from pydantic import BaseModel
from typing import Literal

QuestionCategory = Literal[
    "Lookup/Extraction",
    "Aggregation",
    "Comparison/Change",
    "Transform/Normalize",
    "Statistical Metrics",
    "Modeling/Forecasting"
]
lm = dspy.LM("openrouter/google/gemini-3-flash-preview")
dspy.configure(lm=lm)

class QuestionClassifier(dspy.Signature):
    question: str = dspy.InputField(description="The question to classify")
    category: QuestionCategory = dspy.OutputField(description="The category of the question")

classify = dspy.ChainOfThought(QuestionClassifier)



In [4]:
import dspy
import pandas as pd

df = pd.read_csv("../.dataset/new_runs_base/solved_dataset.csv")

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import dspy
import pandas as pd
from tqdm import tqdm

category_list = []


def classify_question(question):
    """Helper function to classify a single question"""
    return classify(question=question)

# Extract questions as a list
questions = df['question'].tolist()

# Parallelize with ThreadPoolExecutor
# Adjust max_workers based on your API rate limits
max_workers = 5  # Start with 5, increase if API allows

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks and collect futures
    futures = [executor.submit(classify_question, q) for q in questions]
    
    # Process completed tasks with progress bar
    category_list = []
    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying questions"):
        try:
            category = future.result()
            category_list.append(category.category)
        except Exception as e:
            print(f"Error processing question: {e}")
            category_list.append(None)


Classifying questions: 100%|██████████| 246/246 [01:40<00:00,  2.44it/s]


In [8]:
df.category = category_list
df.to_csv("../.dataset/new_runs_base/solved_dataset.csv", index=False)